# Reasoning Agent: KB + Query → True/False
Two front-ends (natural language, direct logic) → one internal representation → classical reasoning
(forward chaining for Horn/Datalog, SOS resolution with unification for general FOL).

In [1]:
import re, time, heapq, itertools, json, os
from collections import defaultdict

In [2]:
# Fresh environment (e.g. Colab): uncomment once
# !pip install -q datasets huggingface_hub pandas pyarrow

In [ ]:
# FOLIO is gated: accept its terms on the dataset page, then uncomment and run once.
# from huggingface_hub import login
# login()

## 1. Terms, substitution, unification
Variable = str starting with `?`. Constant = other str. Compound term = `(name, args_tuple)`.
Literal = `(sign: bool, pred: str, args: tuple)`.

In [3]:
def is_var(t): return isinstance(t, str) and t[:1] == '?'

def walk(t, s):
    while is_var(t) and t in s:
        t = s[t]
    return t

def subst(t, s):
    t = walk(t, s)
    if isinstance(t, tuple):
        return (t[0], tuple(subst(a, s) for a in t[1]))
    return t

def occurs(v, t, s):
    t = walk(t, s)
    if t == v: return True
    if isinstance(t, tuple): return any(occurs(v, a, s) for a in t[1])
    return False

def unify(a, b, s):
    a, b = walk(a, s), walk(b, s)
    if a == b: return s
    if is_var(a):
        if occurs(a, b, s): return None
        s2 = dict(s); s2[a] = b; return s2
    if is_var(b): return unify(b, a, s)
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a[1]) == len(b[1]):
        for x, y in zip(a[1], b[1]):
            s = unify(x, y, s)
            if s is None: return None
        return s
    return None

def term_vars(t, acc):
    if is_var(t): acc.append(t)
    elif isinstance(t, tuple):
        for a in t[1]: term_vars(a, acc)

def term_size(t):
    return 1 + sum(term_size(a) for a in t[1]) if isinstance(t, tuple) else 1

def lit_ground(l):
    acc = []
    for a in l[2]: term_vars(a, acc)
    return not acc

def neg(l): return (not l[0], l[1], l[2])

def fmt_term(t):
    return f"{t[0]}({','.join(fmt_term(a) for a in t[1])})" if isinstance(t, tuple) else str(t)

def fmt_lit(l):
    return ("" if l[0] else "¬") + l[1] + ("(" + ",".join(fmt_term(a) for a in l[2]) + ")" if l[2] else "")

## 2. Forward chaining (semi-naive, predicate-indexed)
Used for Horn / Datalog KBs. `naf=True` gives the closed-world reading of `not` in rule bodies
(negation as failure, evaluated after the monotone part has saturated).

In [4]:
class ForwardChainer:
    def __init__(self, rules, facts, naf=False, max_facts=300000):
        self.rules, self.naf, self.max_facts = rules, naf, max_facts
        self.facts, self.index, self.why = set(), defaultdict(list), {}
        self.queue, self.trig, self.nafrules = [], defaultdict(list), []
        self.domain = sorted({a for body, head in rules for l in body + [head] for a in l[2] if isinstance(a, str) and not is_var(a)} |
                             {a for f in facts for a in f[2] if isinstance(a, str) and not is_var(a)})
        for rid, (body, head) in enumerate(rules):
            if not body:
                if lit_ground(head): self.add(head, ('given', ()))
                continue
            if naf and any(not l[0] for l in body):
                self.nafrules.append(rid)
            else:
                for k, l in enumerate(body):
                    self.trig[(l[0], l[1])].append((rid, k))
        for f in facts: self.add(f, ('fact', ()))

    def add(self, f, why):
        if f in self.facts or len(self.facts) >= self.max_facts: return False
        self.facts.add(f); self.index[(f[0], f[1])].append(f); self.why[f] = why; self.queue.append(f)
        return True

    def _match(self, args, fargs, s):
        if len(args) != len(fargs): return None
        for x, y in zip(args, fargs):
            s = unify(x, y, s)
            if s is None: return None
        return s

    def _join(self, lits, s):
        if not lits:
            yield s; return
        (sg, pr, args), rest = lits[0], lits[1:]
        if sg:
            for f in list(self.index.get((True, pr), ())):
                s2 = self._match(args, f[2], s)
                if s2 is not None: yield from self._join(rest, s2)
        else:
            g = tuple(subst(a, s) for a in args)
            unbound = sorted({a for a in g if is_var(a)})
            if unbound:
                if self.naf:      # closed world: "something" ranges over every known entity
                    for combo in itertools.product(self.domain, repeat=len(unbound)):
                        s2 = dict(s); s2.update(zip(unbound, combo))
                        yield from self._join(lits, s2)
                else:             # open world: only explicit negative facts
                    for f in list(self.index.get((False, pr), ())):
                        s2 = self._match(args, f[2], s)
                        if s2 is not None: yield from self._join(rest, s2)
                return
            if (False, pr, g) in self.facts or (self.naf and (True, pr, g) not in self.facts):
                yield from self._join(rest, s)

    @staticmethod
    def _order(body): return sorted(body, key=lambda l: not l[0])

    def _fire(self, rid, s):
        body, head = self.rules[rid]
        h = (head[0], head[1], tuple(subst(a, s) for a in head[2]))
        if not all(not is_var(a) for a in h[2]): return False
        inst = tuple((l[0], l[1], tuple(subst(a, s) for a in l[2])) for l in body)
        return self.add(h, (rid, inst))

    def _drain(self):
        while self.queue:
            f = self.queue.pop()
            for rid, k in self.trig.get((f[0], f[1]), ()):
                body, _ = self.rules[rid]
                s0 = self._match(body[k][2], f[2], {})
                if s0 is None: continue
                rest = self._order([l for i, l in enumerate(body) if i != k])
                for s in list(self._join(rest, s0)): self._fire(rid, s)

    def run(self):
        while True:
            self._drain()
            if not self.nafrules: break
            added = False
            for rid in self.nafrules:
                body, _ = self.rules[rid]
                for s in list(self._join(self._order(body), {})):
                    added |= self._fire(rid, s)
            if not added and not self.queue: break
        return self

    def holds(self, q):
        if q[0]: return q in self.facts
        return q in self.facts or (self.naf and (True, q[1], q[2]) not in self.facts)

    def trace(self, q, depth=0, seen=None):
        seen = seen if seen is not None else set()
        pad = "  " * depth
        if q not in self.facts:
            return [f"{pad}{fmt_lit(q)}  [not derived" + (", holds by NAF]" if self.holds(q) else "]")]
        r, inst = self.why[q]
        if r in ('fact', 'given'): return [f"{pad}{fmt_lit(q)}  [given]"]
        if q in seen: return [f"{pad}{fmt_lit(q)}  [...]"]
        seen.add(q)
        out = [f"{pad}{fmt_lit(q)}  [rule {r}]"]
        for b in inst: out += self.trace(b, depth + 1, seen)
        return out

## 3. Natural-language front-end (RuleTaker-style English)
Facts: `Bob is not kind.`  `The cat chases the dog.`
Rules: `If someone is young and kind then they are smart.`  `Big, red things are round.`

In [5]:
GENERIC = {'things', 'people', 'animals', 'objects', 'ones', 'someone', 'something', 'everyone', 'everything', 'thing', 'person'}
SUBJ_VARS = {'someone', 'something', 'everyone', 'everything', 'they', 'it', 'people', 'things'}

def ent(text):
    t = text.strip().lower()
    t = re.sub(r'^(the|a|an)\s+', '', t)
    return re.sub(r'\s+', '_', t)

def lemma(v):
    v = v.lower()
    return v[:-1] if v.endswith('s') and not v.endswith('ss') else v

KNOWN_VERBS = {'likes', 'chases', 'eats', 'sees', 'visits', 'needs', 'like', 'chase', 'eat', 'see', 'visit', 'need'}

def term_of(text, pron=None):
    if re.fullmatch(r'X\d*', text.strip()): return '?' + text.strip().lower()
    w = text.strip().lower()
    if w in SUBJ_VARS:
        return pron if (w in ('they', 'it') and pron) else '?x'
    return ent(text)

def parse_statement(text, pron=None):
    """One simple English statement → literal, or None."""
    t = text.strip().rstrip('.').strip()
    if not t: return None
    # copula
    m = re.match(r'^(.+?)\s+(?:is|are)\s+(not\s+)?(?:a\s+|an\s+)?(.+)$', t, re.I)
    if m:
        subj, ng, pred = m.group(1), bool(m.group(2)), m.group(3).strip().lower()
        if re.search(r'\s(the|and)\s', ' ' + pred + ' ') and ' ' in pred:
            return None
        return (not ng, re.sub(r'\s+', '_', pred), (term_of(subj, pron),))
    # does not / do not VERB OBJ
    m = re.match(r'^(.+?)\s+(?:does|do)\s+not\s+(\w+)\s+(?:the\s+)?(.+)$', t, re.I)
    if m:
        return (False, lemma(m.group(2)), (term_of(m.group(1), pron), term_of(m.group(3), pron)))
    # SUBJ VERB OBJ with a known relation word (works with or without "the")
    toks = t.split()
    for i in range(1, len(toks) - 1):
        if toks[i].lower() in KNOWN_VERBS:
            return (True, lemma(toks[i]), (term_of(' '.join(toks[:i]), pron), term_of(' '.join(toks[i + 1:]), pron)))
    # SUBJ VERB the OBJ
    idx = [i for i, w in enumerate(toks) if w.lower() == 'the' and i >= 2]
    if idx:
        i = idx[-1]
        return (True, lemma(toks[i - 1]), (term_of(' '.join(toks[:i - 1]), pron), term_of(' '.join(toks[i + 1:]), pron)))
    if len(toks) >= 3 and toks[-1][0].isupper():
        return (True, lemma(toks[-2]), (term_of(' '.join(toks[:-2]), pron), term_of(toks[-1], pron)))
    if len(toks) == 3 and toks[1].lower().endswith('s'):
        return (True, lemma(toks[1]), (term_of(toks[0], pron), term_of(toks[2], pron)))
    return None

def starts_with_subject(part):
    w = part.strip().split()[0] if part.strip() else ''
    return w.lower() in SUBJ_VARS or w.lower() in ('the', 'a', 'an') or w[:1].isupper()

def subject_text(stmt):
    m = re.match(r'^(.+?)\s+(?:is|are|does|do)\b', stmt, re.I)
    if m: return m.group(1)
    toks = stmt.split()
    for i in range(1, len(toks) - 1):
        if toks[i].lower() in KNOWN_VERBS: return ' '.join(toks[:i])
    idx = [i for i, w in enumerate(toks) if w.lower() == 'the' and i >= 2]
    return ' '.join(toks[:idx[-1] - 1]) if idx else toks[0]

def parse_conj(text, pron=None):
    """'A is x and not y and chases the cat' → list of literals, sharing the first subject."""
    parts = re.split(r'\s+and\s+', text.strip(), flags=re.I)
    lits, subj = [], None
    for p in parts:
        if starts_with_subject(p) and not p.lower().startswith('not '):
            l = parse_statement(p, pron); subj = subject_text(p)
        else:
            if subj is None: return None
            has_obj = re.search(r'\bthe\b', p, re.I) or re.search(r'\s[A-Z]\w+$', p)
            l = parse_statement(f"{subj} {'' if has_obj else 'is '}{p}", pron)
        if l is None: return None
        lits.append(l)
    return lits

def parse_universal(t):
    m = re.match(r'^(?:all|every|each)?\s*(.+?)\s+(?:are|is)\s+(.+)$', t, re.I)
    if not m: return None
    subj, pred = m.group(1).strip(), m.group(2).strip()
    words = subj.lower().replace(',', ' , ').split()
    explicit = t.lower().split()[0] in ('all', 'every', 'each')
    if not (explicit or (words and words[-1] in GENERIC)): return None
    words = [w for w in words if w not in GENERIC and w not in ('all', 'every', 'each', 'and', ',')]
    body, neg_next = [], False
    for w in words:
        if w == 'not': neg_next = True; continue
        body.append((not neg_next, w, ('?x',))); neg_next = False
    head = []
    for p in re.split(r'\s+and\s+|\s*,\s*', pred, flags=re.I):
        p = p.strip().lower()
        if not p: continue
        ng = p.startswith('not ')
        head.append((not ng, re.sub(r'\s+', '_', p[4:] if ng else p), ('?x',)))
    return (body, head) if head else None

def parse_nl_kb(context):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n', context) if s.strip()]
    rules, facts, skipped = [], [], []
    for s in sents:
        t = s.rstrip('.!?').strip()
        m = re.match(r'^if\s+(.+?),?\s+then\s+(.+)$', t, re.I)
        if m:
            body = parse_conj(m.group(1))
            if body is None: skipped.append(s); continue
            first = body[0][2][0]
            head = parse_conj(m.group(2), pron=first)
            if head is None: skipped.append(s); continue
            for h in head: rules.append((body, h))
            continue
        u = parse_universal(t)
        if u:
            body, head = u
            for h in head:
                if body: rules.append((body, h))
            continue
        lits = parse_conj(t)
        if lits is None: skipped.append(s); continue
        facts.extend(lits)
    return rules, facts, skipped

## 3b. Direct-logic front-ends used by RuleTaker's generator: prefix form and Problog
Prefix: `+ ( nice 'lion' )`, `[ + ( cold X ) , + ( kind X ) ] -> + ( nice X )`.
Problog: `1.0::nice(X) :- cold(X), \\+ red(X).`, `query(nice('lion')).`

In [6]:
PTOK = re.compile(r"'[^']*'|->|\[|\]|\(|\)|,|[+-]|[^\s()\[\],]+")

def _parg(tok):
    if tok.startswith("'"): return ent(tok.strip("'"))
    if re.fullmatch(r'[A-Z]\d*', tok): return '?' + tok.lower()
    return ent(tok)

def _ppred(p, arity): return lemma(p) if arity == 2 else p.lower()

def parse_prefix(text):
    """Returns (rules, facts). Statements may be separated by newlines and/or commas."""
    toks = PTOK.findall(text); i = 0; rules, facts = [], []
    def fact():
        nonlocal i
        sign = toks[i] == '+'; i += 1
        assert toks[i] == '('; i += 1
        name = toks[i]; i += 1; args = []
        while toks[i] != ')': args.append(_parg(toks[i])); i += 1
        i += 1
        return (sign, _ppred(name, len(args)), tuple(args))
    while i < len(toks):
        if toks[i] == ',': i += 1; continue
        if toks[i] == '[':
            i += 1; body = []
            while toks[i] != ']':
                if toks[i] == ',': i += 1; continue
                body.append(fact())
            i += 1; assert toks[i] == '->'; i += 1
            rules.append((body, fact()))
        else:
            facts.append(fact())
    return rules, facts

def _patoms(s):
    out = []
    for ng, name, args in re.findall(r"(\\\+\s*)?(\w+)\(([^()]*)\)", s):
        a = tuple(_parg(x.strip()) for x in args.split(','))
        out.append((not ng, _ppred(name, len(a)), a))
    return out

def parse_problog(lines):
    rules, facts = [], []
    for ln in lines:
        ln = ln.strip().rstrip('.').strip()
        if not ln or ln.startswith('query('): continue
        m = re.match(r'^([0-9.]+)::(.*)$', ln)
        if m:
            if float(m.group(1)) == 0.0: continue
            ln = m.group(2)
        if ':-' in ln:
            h, b = ln.split(':-', 1)
            hs = _patoms(h)
            if hs: rules.append((_patoms(b), hs[0]))
        else:
            facts.extend(_patoms(ln))
    return rules, facts

def parse_logic_query(q):
    q = q.strip()
    if re.match(r"^\s*(\[|[+-]\s*\()", q): return parse_prefix(q)[1]
    q = re.sub(r'^\?-\s*', '', q).rstrip('. ')
    m = re.match(r'^query\((.*)\)$', q)
    if m: q = m.group(1)
    return _patoms(q)

def logic_kb_solve(rules, facts, query_lits, naf=True):
    fc = ForwardChainer(rules, facts, naf=naf).run()
    ok = bool(query_lits) and all(fc.holds(l) for l in query_lits)
    return ok, ('forward-chaining', fc.trace(query_lits[0]) if query_lits else ['empty query'])

## 3c. Counterfactual front-end + engine (CounterBench-style questions)
Text → deterministic structural causal model (Boolean equations) → intervene (`do`) → propagate over the DAG.
Handles basic / joint / nested / conditional questions, `A causes B`, `A and B together cause C`, `A or B causes C`, `not A causes B`.

In [7]:
CAUSE_RE = re.compile(r'\s+(?:together\s+)?causes?\s+')

def _lit(tok):
    tok = tok.strip().rstrip('.?,').strip()
    m = re.match(r'^not\s+(.+)$', tok, re.I)
    return (False, m.group(1).strip()) if m else (True, tok)

def _lit_list(text):
    text = re.sub(r'\s+instead of\s+.*$', '', text.strip(), flags=re.I)
    return [_lit(p) for p in re.split(r',\s*(?:and\s+)?|\s+and\s+', text) if p.strip()]

def parse_cf(text):
    """→ (equations, target, interventions, observations, root_default).
    equations: var -> list of (lhs, rhs_positive); lhs = list of OR-terms, each a list of (positive, var) ANDed."""
    t = ' '.join(str(text).split())
    m = re.search(r'We know that\s+(.*)', t)
    if m: body = m.group(1)
    else:
        body = t.split(':', 1)[1] if ':' in t else t
        body = re.sub(r'\s+(?:has|have)\s+(?:a\s+)?direct\s+effects?\s+on\s+', ' causes ', body)
    cut = body.find('. ')
    sent, rest = (body[:cut], body[cut + 2:]) if cut >= 0 else (body.rstrip('.'), '')
    parts = CAUSE_RE.split(sent.strip())
    eqs = defaultdict(list)
    lhs_txt = parts[0]
    for j in range(1, len(parts)):
        seg = parts[j]
        if j < len(parts) - 1:
            rhs_txt, nxt = seg.rsplit(',', 1)
            nxt = re.sub(r'^\s*and\s+', '', nxt)
        else:
            rhs_txt, nxt = seg, None
        lhs = [[_lit(x) for x in re.split(r'\s+and\s+', term)] for term in re.split(r'\s+or\s+', lhs_txt.strip())]
        for pos, v in _lit_list(rhs_txt):
            eqs[v].append((lhs, pos))
        lhs_txt = nxt
    q = re.search(r'Would\s+(?:the\s+)?(.+?)\s+occur(?:\s+if\s+(.+?))?\s*\?', rest, re.I)
    target = q.group(1).strip() if q else None
    interventions = _lit_list(q.group(2)) if q and q.group(2) else []
    for mm in re.finditer(r'(?:Assume|further suppose|suppose)\s+((?:not\s+)?\w+)', rest, re.I):
        interventions.append(_lit(mm.group(1)))
    obs = []
    for mm in re.finditer(r'We observed\s+(.+?)(?:\.|\?|$)', rest):
        obs += _lit_list(mm.group(1))
    return eqs, target, interventions, obs

def cf_solve(text):
    eqs, target, interventions, obs = parse_cf(text)
    if not target or not eqs: return False, ('cf-parse-failed', [str(text)[:80]])
    forced = {v: p for p, v in interventions}
    given = {v: p for p, v in obs}
    memo, trace, busy = {}, [], set()
    def val(v):
        if v in forced: return forced[v]
        if v in memo: return memo[v]
        if v in busy: return False
        if v not in eqs:                       # exogenous variable: observed value, else it occurs
            memo[v] = given.get(v, True); return memo[v]
        busy.add(v)
        out = False
        for lhs, pos in eqs[v]:
            cause = any(all(val(x) == p for p, x in term) for term in lhs)
            out = out or (cause if pos else not cause)
        busy.discard(v); memo[v] = out
        parts_ = []
        for lhs, pos in eqs[v]:
            e = ' OR '.join(' AND '.join(('' if p else 'NOT ') + x for p, x in term) for term in lhs)
            parts_.append(e if pos else f"NOT ({e})")
        trace.append(f"{v} = {' OR '.join(parts_)} = {int(out)}")
        return out
    ans = val(target)
    head = ['do(' + ', '.join(('' if p else 'not ') + v for p, v in interventions) + ')'] if interventions else []
    return ans, ('causal-propagation', head + trace)

def looks_counterfactual(text):
    return bool(re.search(r'\bWould\s+\S+\s+occur\b', str(text))) and bool(re.search(r'\bcauses?\b|direct effect', str(text)))

## 4. Direct-logic front-end: FOL parser → CNF (skolemisation)
Accepts `∀x (Dog(x) → Animal(x))`, `∃`, `¬ ∧ ∨ → ↔ ⊕`, and ASCII `-> <-> & | ~ !`.
Unbound single letters u–z are treated as implicitly universal.

In [8]:
TOK = re.compile(r"\s*(<->|->|∀|∃|¬|∧|∨|→|↔|⟷|⊕|⇒|⇔|≠|=|\(|\)|,|\.|&|\||~|!|[\w']+(?:-(?!>)[\w']+)*)")

def tokenize(s):
    s = s.strip(); pos, out = 0, []
    while pos < len(s):
        m = TOK.match(s, pos)
        if not m:
            if s[pos].isspace(): pos += 1; continue
            raise ValueError(f"bad char {s[pos]!r} in {s!r}")
        out.append(m.group(1)); pos = m.end()
    return out

QUANT = {'∀': 'all', '∃': 'ex', 'forall': 'all', 'exists': 'ex'}
OPS = {'¬': 'not', '~': 'not', '!': 'not'}

class FOLParser:
    def __init__(self, text):
        self.t, self.i = tokenize(text), 0
    def peek(self): return self.t[self.i] if self.i < len(self.t) else None
    def eat(self, x=None):
        tok = self.peek()
        if x is not None and tok != x: raise ValueError(f"expected {x}, got {tok}")
        self.i += 1; return tok
    def parse(self):
        f = self.iff()
        if self.peek() == '.': self.eat()
        if self.peek() is not None: raise ValueError(f"trailing {self.peek()}")
        return f
    def iff(self):
        a = self.imp()
        while self.peek() in ('↔', '<->', '⟷', '⇔'):
            self.eat(); a = ('iff', a, self.imp())
        return a
    def imp(self):
        a = self.disj()
        if self.peek() in ('→', '->', '⇒'):
            self.eat(); return ('imp', a, self.imp())
        return a
    def disj(self):
        a = self.conj()
        while self.peek() in ('∨', '|', '⊕'):
            op = self.eat(); b = self.conj()
            a = ('xor' if op == '⊕' else 'or', a, b)
        return a
    def conj(self):
        a = self.unary()
        while self.peek() in ('∧', '&'):
            self.eat(); a = ('and', a, self.unary())
        return a
    def unary(self):
        tok = self.peek()
        if tok in OPS: self.eat(); return ('not', self.unary())
        if tok in QUANT:
            self.eat(); vs = [self.eat()]
            while self.peek() == ',': self.eat(); vs.append(self.eat())
            while self.peek() and re.match(r'^[a-z][A-Za-z0-9_\']*$', self.peek()) and self.t[self.i + 1:self.i + 2] not in (['('],):
                vs.append(self.eat())
            if self.peek() == '.': self.eat()
            nxt = self.peek()
            body = self.unary() if nxt in ('(', '∀', '∃', '¬', '~', '!') else self.iff()
            for v in reversed(vs): body = (QUANT[tok], v, body)
            return body
        if tok == '(':
            self.eat(); f = self.iff(); self.eat(')'); return f
        name = self.eat()
        if self.peek() == '(':
            self.eat(); args = [self.term()]
            while self.peek() == ',': self.eat(); args.append(self.term())
            self.eat(')')
            if self.peek() in ('=', '≠'):               # f(a) = b
                op = self.eat(); eq = ('atom', '=', ((name, tuple(args)), self.term()))
                return ('not', eq) if op == '≠' else eq
            return ('atom', name, tuple(args))
        if self.peek() in ('=', '≠'):                   # x = y,  x ≠ y
            op = self.eat(); eq = ('atom', '=', (name, self.term()))
            return ('not', eq) if op == '≠' else eq
        return ('atom', name, ())
    def term(self):
        name = self.eat()
        if self.peek() == '(':
            self.eat(); args = [self.term()]
            while self.peek() == ',': self.eat(); args.append(self.term())
            self.eat(')'); return (name, tuple(args))
        return name

def parse_fol(text): return FOLParser(text).parse()

def free_names(f, bound, acc):
    t = f[0]
    if t == 'atom':
        def go(x):
            if isinstance(x, tuple): [go(a) for a in x[1]]
            elif x not in bound: acc.add(x)
        [go(a) for a in f[2]]
    elif t == 'not': free_names(f[1], bound, acc)
    elif t in ('all', 'ex'): free_names(f[2], bound | {f[1]}, acc)
    else: free_names(f[1], bound, acc); free_names(f[2], bound, acc)

_cnt = itertools.count()

def elim(f):
    t = f[0]
    if t == 'atom': return f
    if t == 'not': return ('not', elim(f[1]))
    if t in ('all', 'ex'): return (t, f[1], elim(f[2]))
    a, b = elim(f[1]), elim(f[2])
    if t in ('and', 'or'): return (t, a, b)
    if t == 'imp': return ('or', ('not', a), b)
    if t == 'iff': return ('and', ('or', ('not', a), b), ('or', ('not', b), a))
    if t == 'xor': return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    raise ValueError(t)

def nnf(f, ng=False):
    t = f[0]
    if t == 'atom': return ('lit', not ng, f[1], f[2])
    if t == 'not': return nnf(f[1], not ng)
    if t in ('and', 'or'):
        return ((('or' if t == 'and' else 'and') if ng else t), nnf(f[1], ng), nnf(f[2], ng))
    return ((('ex' if t == 'all' else 'all') if ng else t), f[1], nnf(f[2], ng))

def sub_names(x, env):
    if isinstance(x, tuple): return (x[0], tuple(sub_names(a, env) for a in x[1]))
    return env.get(x, x)

def skolemize(f, env, univ):
    t = f[0]
    if t == 'lit': return ('lit', f[1], f[2], tuple(sub_names(a, env) for a in f[3]))
    if t in ('and', 'or'): return (t, skolemize(f[1], env, univ), skolemize(f[2], env, univ))
    n = next(_cnt)
    if t == 'all':
        v = f"?u{n}"; return skolemize(f[2], {**env, f[1]: v}, univ + [v])
    sk = f"sk{n}"
    term = sk if not univ else (sk, tuple(univ))
    return skolemize(f[2], {**env, f[1]: term}, univ)

MAX_CLAUSES, MAX_TREE = 20000, 20000

def tree_size(f, memo):
    """Expanded node count of a formula whose subtrees may be shared (⊕ / ↔ duplicate them). Linear time."""
    k = id(f)
    if k not in memo:
        if f[0] == 'atom': memo[k] = 1
        elif f[0] == 'not': memo[k] = 1 + tree_size(f[1], memo)
        elif f[0] in ('all', 'ex'): memo[k] = 1 + tree_size(f[2], memo)
        else: memo[k] = 1 + tree_size(f[1], memo) + tree_size(f[2], memo)
    return memo[k]

def to_clauses(f):
    if f[0] == 'lit': return [[(f[1], f[2], f[3])]]
    A, B = to_clauses(f[1]), to_clauses(f[2])
    if f[0] == 'and':
        if len(A) + len(B) > MAX_CLAUSES: raise ValueError('CNF too large')
        return A + B
    if len(A) * len(B) > MAX_CLAUSES: raise ValueError('CNF too large')   # ∨ distributes over ∧: product blow-up
    return [x + y for x in A for y in B]

def clean_clause(c):
    out = []
    for l in c:
        if l not in out: out.append(l)
    for l in out:
        if neg(l) in out: return None   # tautology
    return tuple(out)

def formula_to_clauses(f):
    names = set(); free_names(f, set(), names)
    for nm in sorted(names):
        if re.match(r'^[u-z]\d*$', nm): f = ('all', nm, f)
    f = elim(f)
    if tree_size(f, {}) > MAX_TREE: raise ValueError('formula too large for CNF')
    f = skolemize(nnf(f), {}, [])
    cs = [clean_clause(c) for c in to_clauses(f)]
    return [c for c in cs if c is not None]
def balance_fix(s):
    """FOLIO has premises with unbalanced parentheses. Missing ')' -> append at the end;
    surplus ')' -> open a '(' right after the leading quantifier prefix (or at the start)."""
    s = s.strip(); d = s.count('(') - s.count(')')
    if d > 0: return s + ')' * d
    if d < 0:
        m = re.match(r'^((?:[∀∃]\s*\w+[\s,]*)+)', s)
        return (s[:m.end()] + '(' * (-d) + s[m.end():]) if m else '(' * (-d) + s
    return s

def parse_fol_robust(text):
    """→ (formula, repaired?). Raises only if even the repaired text does not parse."""
    try: return parse_fol(text), False
    except Exception: return parse_fol(balance_fix(text)), True


## 5. Resolution prover (set-of-support, given-clause loop, factoring, time budget)

In [9]:
def rename(clause, tag):
    def r(t):
        if is_var(t): return t + tag
        if isinstance(t, tuple): return (t[0], tuple(r(a) for a in t[1]))
        return t
    return tuple((l[0], l[1], tuple(r(a) for a in l[2])) for l in clause)

def normalize(clause):
    lits = sorted(set(clause), key=repr)
    m = {}
    def r(t):
        if is_var(t):
            if t not in m: m[t] = f"?{len(m)}"
            return m[t]
        if isinstance(t, tuple): return (t[0], tuple(r(a) for a in t[1]))
        return t
    lits = [(l[0], l[1], tuple(r(a) for a in l[2])) for l in lits]
    return tuple(sorted(lits, key=repr))

def apply_lit(l, s): return (l[0], l[1], tuple(subst(a, s) for a in l[2]))

class Prover:
    def __init__(self, axioms, goal, time_limit=5.0, max_clauses=40000, max_lits=8, max_term=8):
        self.t0, self.tl, self.maxc, self.maxl, self.maxt = time.perf_counter(), time_limit, max_clauses, max_lits, max_term
        self.seen, self.index, self.sos, self.n = set(), defaultdict(list), [], itertools.count()
        self.parents = {}
        for c in axioms: self._add_usable(normalize(c))
        for c in goal:
            c = normalize(c)
            if c not in self.seen: self.seen.add(c); self._push(c, ('goal',))
        self.generated = 0

    def _push(self, c, why):
        self.parents[c] = why
        heapq.heappush(self.sos, (len(c) + sum(term_size(a) for l in c for a in l[2]) * 0.1, next(self.n), c))

    def _add_usable(self, c):
        if c in self.seen: return
        self.seen.add(c); self.parents[c] = ('axiom',)
        self._index(c)

    def _index(self, c):
        for j, l in enumerate(c): self.index[(l[0], l[1])].append((c, j))

    def _ok(self, c):
        return len(c) <= self.maxl and all(term_size(a) <= self.maxt for l in c for a in l[2])

    def _factors(self, c):
        for i in range(len(c)):
            for j in range(i + 1, len(c)):
                if c[i][0] == c[j][0] and c[i][1] == c[j][1]:
                    s = {}
                    for x, y in zip(c[i][2], c[j][2]):
                        s = unify(x, y, s)
                        if s is None: break
                    if s is not None:
                        yield tuple(apply_lit(l, s) for k, l in enumerate(c) if k != j)

    def prove(self):
        while self.sos:
            if time.perf_counter() - self.t0 > self.tl or self.generated > self.maxc: return False
            _, _, g = heapq.heappop(self.sos)
            if not g: return True
            new = list(self._factors(g))
            ga = rename(g, 'a')
            for i, l in enumerate(ga):
                for c, j in list(self.index.get((not l[0], l[1]), ())):
                    cb = rename(c, 'b'); l2 = cb[j]
                    if len(l2[2]) != len(l[2]): continue
                    s = {}
                    for x, y in zip(l[2], l2[2]):
                        s = unify(x, y, s)
                        if s is None: break
                    if s is None: continue
                    new.append(tuple(apply_lit(m, s) for k, m in enumerate(ga) if k != i) +
                               tuple(apply_lit(m, s) for k, m in enumerate(cb) if k != j))
            self._add_usable_given(g)
            for r in new:
                r = clean_clause(r)
                if r is None: continue
                r = normalize(r)
                if not self._ok(r) or r in self.seen: continue
                self.seen.add(r); self.generated += 1
                if not r: return True
                self._push(r, ('res', g))
        return False

    def _add_usable_given(self, g):
        self._index(g)

## 6. Router + solvers

In [10]:
LOGIC_HINT = re.compile(r"[∀∃∧∨→↔¬⊕]|->|<->|\b[A-Z][A-Za-z0-9_]*\([^)]*\)")

def looks_logic(text): return bool(LOGIC_HINT.search(text))

def as_lines(x):
    if isinstance(x, (list, tuple)): return [str(i) for i in x if str(i).strip()]
    return [l for l in str(x).split('\n') if l.strip()]

def clauses_to_horn(clauses):
    """If every clause is definite (exactly one positive literal) and function-free → (facts, rules)."""
    facts, rules = [], []
    for c in clauses:
        pos = [l for l in c if l[0]]
        if len(pos) != 1: return None
        if any(isinstance(a, tuple) for l in c for a in l[2]): return None
        body = [(True, l[1], l[2]) for l in c if not l[0]]
        if body: rules.append((body, pos[0]))
        elif lit_ground(pos[0]): facts.append(pos[0])
        else: return None
    return facts, rules

def _has_eq(clauses): return any(l[1] == '=' for c in clauses for l in c)

def fol_solve(premises, conclusion, time_limit=5.0, trace=False):
    axioms, skipped, repaired = [], 0, 0
    for p in as_lines(premises):                 # one bad premise no longer kills the whole example
        try:
            f, rep = parse_fol_robust(p); axioms += formula_to_clauses(f); repaired += rep
        except Exception:
            skipped += 1
    q, rep = parse_fol_robust(conclusion.strip()); repaired += rep
    note = f"(premises repaired: {repaired}, skipped: {skipped})"
    goal_pos = formula_to_clauses(q)
    if len(goal_pos) == 1 and len(goal_pos[0]) == 1 and goal_pos[0][0][0] and lit_ground(goal_pos[0][0]):
        horn = clauses_to_horn(axioms)
        if horn:
            fc = ForwardChainer(horn[1], horn[0]).run()
            ok = goal_pos[0][0] in fc.facts
            if ok: return True, ('forward-chaining', fc.trace(goal_pos[0][0]) + [note])
    goal = formula_to_clauses(('not', q))
    if _has_eq(axioms) or _has_eq(goal):         # equality: reflexivity + symmetry
        axioms = axioms + [[(True, '=', ('?e', '?e'))], [(False, '=', ('?a', '?b')), (True, '=', ('?b', '?a'))]]
    p = Prover(axioms, goal, time_limit=time_limit)
    ok = p.prove()
    return ok, ('resolution', [f"refutation {'found' if ok else 'not found'} after {p.generated} clauses", note])

def nl_solve(context, query, naf=True):
    rules, facts, skipped = parse_nl_kb(context)
    q = parse_conj(query.rstrip('.!? '))
    if not q: return False, ('nl-parse-failed', [query])
    fc = ForwardChainer(rules, facts, naf=naf).run()
    ok = all(fc.holds(l) for l in q)
    return ok, ('forward-chaining', fc.trace(q[0]) + ([f"(skipped {len(skipped)} unparsed sentences)"] if skipped else []))

def is_prefix(text): return bool(re.search(r"(^|\n)\s*(\[|[+-]\s*\()", text))
def is_problog(text):
    if re.search(r"::|:-|^\s*(query\(|\?-)", text, re.M): return True
    lines = [l for l in text.split('\n') if l.strip()]
    return bool(lines) and all(re.match(r"^\s*[a-z]\w*\(.*\)\s*\.?\s*$", l) for l in lines)

def solve(context, query, naf=True, time_limit=5.0):
    """Returns (bool, (engine, trace))."""
    try:
        ctx_text = '\n'.join(as_lines(context))
        if looks_counterfactual(ctx_text + ' ' + str(query)):
            return cf_solve(ctx_text + ' ' + str(query) if ctx_text.strip() != str(query).strip() else ctx_text)
        if is_prefix(ctx_text) or is_prefix(str(query)):
            r, f = parse_prefix(ctx_text); return logic_kb_solve(r, f, parse_logic_query(str(query)), naf)
        if is_problog(ctx_text) or is_problog(str(query)):
            r, f = parse_problog(as_lines(ctx_text)); return logic_kb_solve(r, f, parse_logic_query(str(query)), naf)
        if looks_logic(ctx_text) or looks_logic(str(query)):
            return fol_solve(context, str(query), time_limit=time_limit)
        return nl_solve(ctx_text, str(query), naf=naf)
    except Exception as e:
        return False, ('error', [repr(e)])

## 7. Data loading + evaluation — everything comes from Hugging Face
| Dataset | HF repo | Split used | Columns |
|---|---|---|---|
| RuleTaker | `hitachi-nlp/ruletaker` | `dev` / `test` | context, question, label, config |
| CounterBench | `CounterBench/CounterBench` | (question files) | question, answer |
| FOLIO | `yale-nlp/FOLIO` | `validation` | premises-FOL, conclusion-FOL, label |

Column names are auto-detected. Call `peek(df)` once per dataset to confirm. A local `.csv/.json/.jsonl/.parquet`
path is still accepted by `load_df` so the private test file can be dropped in later.

In [11]:
import pandas as pd
from datasets import load_dataset

HF_REPOS = {
    'ruletaker':    'hitachi-nlp/ruletaker',
    'counterbench': 'CounterBench/CounterBench',
    'folio':        'yale-nlp/FOLIO',
}

CTX_COLS = ['premises-FOL', 'given_info', 'context', 'theory', 'premises', 'knowledge_base', 'kb', 'story', 'logic']
Q_COLS   = ['conclusion-FOL', 'question', 'statement', 'conclusion', 'query', 'hypothesis']
LAB_COLS = ['label', 'answer', 'flag', 'gold']
ID_COLS  = ['id', 'ID', 'example_id', 'idx']

def pick(df, cands):
    for c in cands:
        if c in df.columns: return c
    return None

def to_bool(x):
    return str(x).strip().lower() in ('true', '1', 'yes', 'entailment', 'entailed', 't')

def _sample(df, n, seed=0):
    """Random (not head) sample so every depth / question type is represented."""
    return df.sample(n=n, random_state=seed).reset_index(drop=True) if n and n < len(df) else df.reset_index(drop=True)

def load_hf(repo, split, n=None, seed=0, **kw):
    """Load one split from the Hugging Face Hub; falls back to the other common split names."""
    order = [split] + [s for s in ('dev', 'validation', 'test', 'train') if s != split]
    for sp in order:
        try:
            df = load_dataset(repo, split=sp, **kw).to_pandas()
            print(f"loaded {repo} [{sp}]  rows={len(df)}")
            return _sample(df, n, seed)
        except (ValueError, KeyError) as e:
            last = e
    raise last

def load_ruletaker(split='dev', n=None, seed=0, depth=None):
    """RuleTaker from HF. `depth` filters the `config` column, e.g. depth='depth-3'."""
    df = load_hf(HF_REPOS['ruletaker'], split)
    if depth is not None:
        df = df[df['config'].astype(str).str.contains(str(depth), regex=False)]
    return _sample(df, n, seed)

def load_folio(split='validation', n=None, seed=0):
    return load_hf(HF_REPOS['folio'], split, n, seed)

def load_counterbench(n=None, seed=0, repo=None):
    """The repo mixes metadata files (meta_model_*.json) with question files, so load_dataset() can fail.
    Try load_dataset first, otherwise download only the question files and concatenate them."""
    repo = repo or HF_REPOS['counterbench']
    try:
        df = load_dataset(repo, split='train').to_pandas()
    except Exception:
        from huggingface_hub import list_repo_files, hf_hub_download
        files = [f for f in list_repo_files(repo, repo_type="dataset")
                 if f.endswith(('.json', '.jsonl', '.csv', '.parquet')) and 'meta_model' not in f]
        print("question files:", files)
        dfs = []
        for f in files:
            p = hf_hub_download(repo, f, repo_type="dataset")
            if p.endswith('.csv'): d = pd.read_csv(p)
            elif p.endswith('.parquet'): d = pd.read_parquet(p)
            else:
                try: raw = json.load(open(p))
                except ValueError: raw = [json.loads(l) for l in open(p) if l.strip()]
                if isinstance(raw, dict): raw = list(raw.values())
                d = pd.DataFrame(raw)
            dfs.append(d)
        df = pd.concat(dfs, ignore_index=True)
    print(f"loaded {repo}  rows={len(df)}")
    return _sample(df, n, seed)

def load_df(src, n=None, seed=0):
    """Local file (private test set) or a Hugging Face repo id."""
    if os.path.exists(src):
        if src.endswith('.csv'): df = pd.read_csv(src)
        elif src.endswith('.jsonl'): df = pd.read_json(src, lines=True)
        elif src.endswith('.json'): df = pd.read_json(src)
        elif src.endswith('.parquet'): df = pd.read_parquet(src)
        else: raise ValueError(src)
        return _sample(df, n, seed)
    return load_hf(src, 'test', n, seed)

def peek(df, label_col=None):
    print(df.columns.tolist()); print(df.iloc[0].to_dict())
    lab = label_col or pick(df, LAB_COLS)
    if lab: print("labels:", df[lab].value_counts().to_dict())
    if 'config' in df.columns: print("configs:", df['config'].value_counts().to_dict())

def evaluate(df, naf=True, ctx=None, q=None, lab=None, by=None, show_fail=3, time_limit=5.0):
    ctx, q, lab = ctx or pick(df, CTX_COLS), q or pick(df, Q_COLS), lab or pick(df, LAB_COLS)
    if ctx is None and q is not None: ctx = q
    if None in (ctx, q, lab):
        raise ValueError(f"could not detect columns (ctx={ctx}, q={q}, label={lab}); got {df.columns.tolist()}. Pass ctx=, q=, lab=.")
    ok = n = 0; engines = defaultdict(int); fails = []; grp = defaultdict(lambda: [0, 0])
    t0 = time.perf_counter()
    for _, r in df.iterrows():
        pred, (eng, _) = solve(r[ctx], r[q], naf=naf, time_limit=time_limit)
        engines[eng] += 1; n += 1
        hit = pred == to_bool(r[lab]); ok += hit
        if by and by in df.columns: g = grp[str(r[by])]; g[0] += hit; g[1] += 1
        if not hit and len(fails) < show_fail: fails.append((r[ctx], r[q], r[lab], pred, eng))
    dt = time.perf_counter() - t0
    print(f"accuracy {100*ok/n:.2f}%  ({ok}/{n})  time {dt:.2f}s  engines {dict(engines)}")
    for k in sorted(grp): print(f"   {k:<14} {100*grp[k][0]/grp[k][1]:6.2f}%  ({grp[k][0]}/{grp[k][1]})")
    return fails

def predict_df(df, out='submission.csv', naf=True, ctx=None, q=None, idc=None, time_limit=5.0):
    ctx, q, idc = ctx or pick(df, CTX_COLS), q or pick(df, Q_COLS), idc or pick(df, ID_COLS)
    if ctx is None: ctx = q
    t0 = time.perf_counter(); rows = []
    for k, r in df.reset_index(drop=True).iterrows():
        pred, _ = solve(r[ctx], r[q], naf=naf, time_limit=time_limit)
        rows.append({'id': r[idc] if idc else k, 'label': bool(pred)})
    pd.DataFrame(rows).to_csv(out, index=False)
    print(f"wrote {out} ({len(rows)} rows) in {time.perf_counter()-t0:.2f}s")

def predict_file(path, out='submission.csv', **kw): predict_df(load_df(path), out=out, **kw)

def show_trace(context, query, naf=True):
    ok, (eng, tr) = solve(context, query, naf=naf)
    print(f"{query!r} → {ok}   [{eng}]"); print('\n'.join(tr))

## 8. Demo + reasoning traces (use these in the report)

In [12]:
show_trace("Bob is young. Bob is kind. If someone is young, then they are smart. If someone is smart and kind, then they are successful.", "Bob is successful.")
print()
show_trace(["∀x (Dog(x) → Animal(x))", "∀x (Animal(x) → ∃y Eats(x,y))", "Dog(rex)"], "∃y Eats(rex,y)")

'Bob is successful.' → True   [forward-chaining]
successful(bob)  [rule 1]
  smart(bob)  [rule 0]
    young(bob)  [given]
  kind(bob)  [given]

'∃y Eats(rex,y)' → True   [resolution]
refutation found after 3 clauses
(premises repaired: 0, skipped: 0)


## 9. Public-set evaluation (all from Hugging Face)
`n` = number of rows sampled at random (seed 0). Per-`config` accuracy shows each RuleTaker depth separately.

In [13]:
rt = load_ruletaker('dev', n=3000)
peek(rt)

loaded hitachi-nlp/ruletaker [dev]  rows=75872
['context', 'question', 'label', 'config']
{'context': 'Bob is furry. Bob is green. Bob is quiet. Bob is smart. Bob is young. Charlie is furry. Charlie is green. Charlie is quiet. Charlie is red. Charlie is smart. Charlie is white. Charlie is young. If something is furry then it is quiet. Furry things are smart. If something is smart and white then it is young. White, smart things are quiet. Quiet things are red. If Bob is green and Bob is red then Bob is white.', 'question': 'Bob is red.', 'label': 'entailment', 'config': 'depth-2'}
labels: {'not entailment': 1510, 'entailment': 1490}
configs: {'depth-3ext-NatLang': 559, 'depth-3ext': 472, 'depth-2': 386, 'depth-1': 381, 'depth-0': 361, 'depth-5': 350, 'depth-3': 341, 'NatLang': 150}


### RuleTaker — closed-world (`naf=True`) vs open-world (`naf=False`)

In [14]:
for naf in (True, False):
    print("naf =", naf)
    fails = evaluate(rt, naf=naf, by='config')

naf = True
accuracy 100.00%  (3000/3000)  time 1.81s  engines {'forward-chaining': 3000}
   NatLang        100.00%  (150/150)
   depth-0        100.00%  (361/361)
   depth-1        100.00%  (381/381)
   depth-2        100.00%  (386/386)
   depth-3        100.00%  (341/341)
   depth-3ext     100.00%  (472/472)
   depth-3ext-NatLang 100.00%  (559/559)
   depth-5        100.00%  (350/350)
naf = False
accuracy 74.93%  (2248/3000)  time 1.69s  engines {'forward-chaining': 3000}
   NatLang         80.00%  (120/150)
   depth-0         76.18%  (275/361)
   depth-1         75.33%  (287/381)
   depth-2         74.35%  (287/386)
   depth-3         71.26%  (243/341)
   depth-3ext      75.21%  (355/472)
   depth-3ext-NatLang  74.60%  (417/559)
   depth-5         75.43%  (264/350)


### CounterBench and FOLIO (run `peek` first; label columns are auto-detected)

In [15]:
cb = load_counterbench(n=500); peek(cb)
fails_cb = evaluate(cb, by='type', show_fail=5)
for ctx, q, lab, pred, eng in fails_cb:
    print('-' * 60); print(ctx); print('Q:', q, '| gold:', lab, '| pred:', pred, '|', eng)

question files: ['data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json']
loaded CounterBench/CounterBench  rows=1200
['question', 'given_info', 'answer', 'type', 'question_id', 'meta']
{'question': 'Would Jext occur if not Fizo instead of Fizo?', 'given_info': 'We know that Fizo causes not Blorn, Blorn causes Plim, Plim causes Quaz, Quaz and Plim together cause Skul, Skul causes Triv, Triv causes Yex, and Yex and Skul together cause Jext.', 'answer': 'yes', 'type': 'basic', 'question_id': '639', 'meta': {'story_id': 'nonsense_3', 'rung': 3, 'query_type': 'det-counterfactual', 'model_id': 198, 'graph_id': 'graph8'}}
labels: {'no': 258, 'yes': 242}
accuracy 94.60%  (473/500)  time 0.12s  engines {'causal-propagation': 500}
   backdoor        86.02%  (80/93)
   basic          100.00%  (105/105)
   conditional     92.78%  (90/97)
   joint           95.45%  (105/110)
   nested          97.89%  (93/95)
------------------------------------------------------------
We know that Glent c

In [18]:
fo = load_folio('validation'); peek(fo)
fails_fo = evaluate(fo, time_limit=1.0)   # 1 s proof budget per example; FOLIO 'Uncertain' counts as False

loaded yale-nlp/FOLIO [validation]  rows=203
['story_id', 'premises', 'premises-FOL', 'conclusion', 'conclusion-FOL', 'label', 'example_id']
{'story_id': 380, 'premises': 'People in this club who perform in school talent shows often attend and are very engaged with school events.\nPeople in this club either perform in school talent shows often or are inactive and disinterested community members.\nPeople in this club who chaperone high school dances are not students who attend the school.\nAll people in this club who are inactive and disinterested members of their community chaperone high school dances.\nAll young children and teenagers in this club who wish to further their academic careers and educational opportunities are students who attend the school. \nBonnie is in this club and she either both attends and is very engaged with school events and is a student who attends the school or is not someone who both attends and is very engaged with school events and is not a student who att

## 10. Final run: accuracy, inference time, submission.csv

In [20]:
NAF = True                                      # pick from the comparison above
test = load_ruletaker('test', n=5000)           # public HF test split while developing
_ = evaluate(test, naf=NAF, by='config')            # prints accuracy + total time

loaded hitachi-nlp/ruletaker [test]  rows=151911
accuracy 99.92%  (4996/5000)  time 3.65s  engines {'forward-chaining': 5000}
   NatLang        100.00%  (245/245)
   depth-0        100.00%  (560/560)
   depth-1        100.00%  (584/584)
   depth-2        100.00%  (587/587)
   depth-3         99.82%  (554/555)
   depth-3ext      99.75%  (789/791)
   depth-3ext-NatLang  99.90%  (1036/1037)
   depth-5        100.00%  (641/641)


In [21]:
PRIVATE_TEST = None   # set to the private test file's path, e.g. 'private_test.csv'

if PRIVATE_TEST:
    predict_file(PRIVATE_TEST, out="submission.csv", naf=NAF)   # add ctx=, q=, idc= if column detection fails
else:
    predict_df(test, out="submission.csv", naf=NAF)             # public-test sample

wrote submission.csv (5000 rows) in 3.47s


In [ ]:
# Colab only: download the file
# from google.colab import files
# files.download("submission.csv")